###  This notebook builds a complete Bronze–Silver–Gold data pipeline in Databricks using the Iris dataset and implements Slowly Changing Dimension Type-2 (SCD-2) in the Gold layer.

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
spark.sql("CREATE DATABASE IF NOT EXISTS gold")

In [0]:
df_bronze = spark.read.csv("/Volumes/workspace/default/source/Iris.csv", header=True, inferSchema=True)
df_bronze.write.mode("append").saveAsTable("bronze.raw_data")

In [0]:
from pyspark.sql.functions import *

bronze_df = spark.table("bronze.raw_data")

silver_df = (
    bronze_df
    .withColumnRenamed("sepal.length","sepal_length")
    .withColumnRenamed("sepal.width","sepal_width")
    .withColumnRenamed("petal.length","petal_length")
    .withColumnRenamed("petal.width","petal_width")
    .withColumn("sepal_length", col("sepal_length").cast("double"))
    .withColumn("sepal_width", col("sepal_width").cast("double"))
    .withColumn("petal_length", col("petal_length").cast("double"))
    .withColumn("petal_width", col("petal_width").cast("double"))
    .fillna({"sepal_length":0.0,"sepal_width":0.0,"petal_length":0.0,"petal_width":0.0})
    .withColumn("sepal_length_sq", round(col("sepal_length")*col("sepal_length"),2))
    .withColumn("variety", upper(trim(col("variety"))))
)

silver_df.write.mode("append").saveAsTable("silver.transformed_data")


In [0]:
silver_base = spark.table("silver.transformed_data")

gold_business_df = silver_base.groupBy("variety").agg(
    round(avg("sepal_length"),2).alias("avg_sepal_length"),
    round(avg("sepal_width"),2).alias("avg_sepal_width"),
    round(avg("petal_length"),2).alias("avg_petal_length"),
    round(avg("petal_width"),2).alias("avg_petal_width"),
    max("sepal_length_sq").alias("max_sepal_length_sq")
)


In [0]:
scd_cols = ["hash_value","valid_from","valid_to","valid_flag","updated"]
business_cols = [c for c in gold_business_df.columns if c not in scd_cols]

incoming_gold = gold_business_df.withColumn(
    "hash_value",
    sha2(concat_ws("||", *business_cols), 256)
).withColumn("valid_from", date_format(current_timestamp(),"dd-MM-yyyy HH:mm:ss")) \
 .withColumn("valid_to", lit("31-12-9999 00:00:00")) \
 .withColumn("valid_flag", lit("Y")) \
 .withColumn("updated", lit(None).cast("string"))

In [0]:
incoming_gold.write.format("delta").mode("append").saveAsTable("gold.iris_gold_mart")

In [0]:
current_gold = spark.table("gold.iris_gold_mart").filter(col("valid_flag")=="Y")

In [0]:
changed_df = incoming_gold.join(current_gold, "variety", "left") \
.filter(current_gold.variety.isNull() | (incoming_gold.hash_value != current_gold.hash_value))

In [0]:
expired_df = current_gold.join(changed_df, "variety") \
.withColumn("valid_to", date_format(current_timestamp(),"dd-MM-yyyy HH:mm:ss")) \
.withColumn("updated", date_format(current_timestamp(),"dd-MM-yyyy HH:mm:ss")) \
.withColumn("valid_flag", lit("N"))

In [0]:
new_records_df = changed_df.select(incoming_gold.columns)

In [0]:
final_gold = (
    spark.table("gold.iris_gold_mart").filter(col("valid_flag")=="N")
    .unionByName(expired_df)
    .unionByName(new_records_df)
)

final_gold.write.format("delta").mode("append").saveAsTable("gold.iris_gold_mart")